In [ ]:
"""
This version is for the fixed-temperature segment/operating-strategy study.

Applied fixes:
1. Shorter representative segment-study horizon:
       HORIZON_HOURS = 1680
   Use 8000 only later for final headline cases after selecting a converged segment count.

2. More realistic Gurobi settings:
       TimeLimit = 600
       MIPGap = 0.005
       MIPFocus = 1
       Threads = 0

3. Records MIPGap and Converged in sensitivity_results and computational_results.

4. Prints warnings for no feasible solution and for high-gap/unreliable cases.

5. Plot functions exclude unconverged rows and use np.nan for missing cases,
   instead of plotting fake zeros.

6. Enables terminal storage fairness:
       e[T - 1] >= e_0
"""

import os
os.environ["GRB_LICENSE_FILE"] = "/Users/mohamedamr/Desktop/Program_2026/Model_Gur/gurobi.lic"

import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


# ============================================================
# User paths
# ============================================================
BASE_DIR = "/Users/mohamedamr/Desktop/Program_2026/Model_Gur"

EFFICIENCY_FILE = f"{BASE_DIR}/power_eta_multi_T_p_55.csv"

WIND_FILE = f"{BASE_DIR}/ninja_wind_55.6761_12.5683_corrected (2).csv"

ENERGY_CHARTS_FILE = (
    f"{BASE_DIR}/energy-charts_Electricity_production_and_spot_prices_in_Denmark_in_2019 (2).csv"
)

MERGED_TIME_FILE = f"{BASE_DIR}/merged_denmark_wind_price_2019.csv"


# ============================================================
# Run controls
# ============================================================
# Segment-convergence study should use a representative window, not full year.
# For full-year headline runs, set this back to 8000 after choosing a converged segment count.
HORIZON_HOURS = 8760
SELECTED_TEMPS = [55, 70, 75, 85]

# Only requested minimum-load case
P_MIN_VALUES = [450]

# Only two operating strategies
OPERATING_CASES = ["ON_OFF_STANDBY"]

OPERATING_CASE_LABELS = {
    "ON_OFF": "on/off",
    "ON_OFF_STANDBY": "on/off/standby"
}

# Case/segment structure
CASE_SEGMENT_OPTIONS = {
    "ON_OFF_STANDBY": [12]
}

# Segment section in final figure uses on/off/standby because it has 1,2,4,8,12
BASE_OPERATING_CASE_FOR_SEGMENT_SECTION = "ON_OFF_STANDBY"

# Wind plant scaling
P_RE_CAPACITY_KW = 1500
SCALE_NINJA_TO_CAPACITY = False

# Price area
PRICE_ZONE = "DK1"


# ============================================================
# Plot controls
# ============================================================
PLOT_TEMPERATURE = 85
PLOT_P_MIN_ON = 450

PLOT_SEGMENTS = [12]
PLOT_OPERATING_CASES = ["ON_OFF_STANDBY"]
STATE_COMPARISON_SEGMENTS = [12]

LOSS_PLOT_PNG = f"{BASE_DIR}/loss_comparison_on_off_vs_standby.png"
LOSS_PLOT_PDF = f"{BASE_DIR}/loss_comparison_on_off_vs_standby.pdf"
PROFIT_PLOT_PNG = f"{BASE_DIR}/profit_comparison_on_off_vs_standby.png"
PROFIT_PLOT_PDF = f"{BASE_DIR}/profit_comparison_on_off_vs_standby.pdf"


# ============================================================
# Output files
# ============================================================
sensitivity_out = f"{BASE_DIR}/minimum_load_300_on_off_standby_results.csv"
hourly_out = f"{BASE_DIR}/minimum_load_300_on_off_standby_hourly_results.csv"
computational_out = f"{BASE_DIR}/computational_aspects_on_off_standby.csv"
computational_latex_out = f"{BASE_DIR}/computational_aspects_on_off_standby.tex"
temperature_table_out = f"{BASE_DIR}/tableII_temperature_sensitivity_on_off_standby_12seg.csv"


# ============================================================
# Helper: read Renewables.ninja CSV robustly
# ============================================================
def read_renewables_ninja_csv(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    header_row = None

    for idx, line in enumerate(lines):
        cols = [c.strip().strip('"') for c in line.split(",")]
        if "time" in cols and "electricity" in cols:
            header_row = idx
            break

    if header_row is None:
        raise ValueError(
            "Could not find a header row containing 'time' and 'electricity' in the Ninja CSV."
        )

    df = pd.read_csv(path, skiprows=header_row)
    df.columns = [str(c).strip().strip('"') for c in df.columns]

    return df


# ============================================================
# Helper: read Energy-Charts CSV robustly
# ============================================================
def read_energy_charts_csv(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8-sig") as f:
        lines = f.readlines()

    header_row = None

    for idx, line in enumerate(lines):
        if "Date" in line and "Day Ahead Auction" in line:
            header_row = idx
            break

    if header_row is None:
        raise ValueError("Could not find Energy-Charts header row.")

    skiprows = list(range(header_row)) + [header_row + 1]

    df = pd.read_csv(path, skiprows=skiprows)
    df.columns = [str(c).strip().strip('"') for c in df.columns]

    return df


# ============================================================
# Operating-strategy constraints
# ============================================================
def add_operating_case_constraints(m, t, case_name, y_on, y_stb):
    """
    Operating strategy definitions:

    ON_OFF:
        On/off operation only.
        Standby is disabled.

    ON_OFF_STANDBY:
        On/off/standby operation.
        Standby is allowed only if the previous state was on or standby.
    """

    if case_name == "ON_OFF":
        m.addConstr(
            y_stb[t] == 0,
            name=f"on_off_no_standby_{t}"
        )

    elif case_name == "ON_OFF_STANDBY":
        if t == 0:
            m.addConstr(
                y_stb[t] == 0,
                name="standby_initial"
            )
        else:
            m.addConstr(
                y_stb[t] <= y_on[t - 1] + y_stb[t - 1],
                name=f"standby_logic_{t}"
            )

    else:
        raise ValueError(f"Unknown operating strategy: {case_name}")


# ============================================================
# Computational-aspects helpers
# ============================================================
def count_binary_variables(model):
    return sum(1 for v in model.getVars() if v.VType == GRB.BINARY)


def format_binary_count(binary_count, horizon_hours):
    if horizon_hours > 0 and binary_count % horizon_hours == 0:
        coefficient = binary_count // horizon_hours
        return f"{coefficient}×{horizon_hours}"
    return str(binary_count)


def print_computational_aspects_table(
    computational_df: pd.DataFrame,
    temperature_c: float = 85,
    p_min_on_kw: float = 450,
    case_order=None,
    output_csv=None,
    output_latex=None
):
    if case_order is None:
        case_order=[
            "on/off-1",
            "on/off-8",
            "on/off/standby-1",
            "on/off/standby-8"
        ]

    df = computational_df.copy()

    df = df[
        (df["Temperature_C"] == temperature_c) &
        (df["P_min_on_kW"] == p_min_on_kw)
    ].copy()

    if df.empty:
        print("\nNo computational results available for the selected table case.")
        return pd.DataFrame()

    df["Case"] = df["Case"].astype(str)

    df = df[df["Case"].isin(case_order)].copy()

    df["Case_order"] = df["Case"].apply(
        lambda x: case_order.index(x) if x in case_order else 999
    )

    df = df.sort_values("Case_order").copy()

    table_df = df[
        [
            "Case",
            "Computational_time_s",
            "Binary_variables_formatted"
        ]
    ].copy()

    table_df.rename(
        columns={
            "Computational_time_s": "Computational time [s]",
            "Binary_variables_formatted": "No. of binary variables"
        },
        inplace=True
    )

    table_df["Computational time [s]"] = table_df["Computational time [s]"].map(
        lambda x: f"{x:,.1f}"
    )

    print("\nTABLE II")
    print("COMPUTATIONAL ASPECTS\n")

    print(
        table_df.to_string(
            index=False,
            justify="center"
        )
    )

    if output_csv is not None:
        table_df.to_csv(output_csv, index=False)
        print(f"\nComputational aspects CSV saved to:\n{output_csv}")

    if output_latex is not None:
        latex_text = table_df.to_latex(
            index=False,
            escape=False,
            column_format="lcc",
            caption="Computational aspects.",
            label="tab:computational_aspects"
        )

        with open(output_latex, "w", encoding="utf-8") as f:
            f.write(latex_text)

        print(f"\nComputational aspects LaTeX table saved to:\n{output_latex}")

    return table_df


# ============================================================
# Final loss-comparison plot
# ============================================================
def plot_loss_comparison_on_off_standby(
    sensitivity_df: pd.DataFrame,
    output_png: str,
    output_pdf: str,
    temperature_c: float = 85,
    p_min_on_kw: float = 450,
    segment_list=(1, 2, 4, 8, 12),
    operating_cases=("ON_OFF", "ON_OFF_STANDBY"),
    state_segments=(1, 12),
    base_operating_case_for_segment_section="ON_OFF_STANDBY"
):
    df = sensitivity_df.copy()

    df = df[
        (df["Temperature_C"] == temperature_c) &
        (df["P_min_on_kW"] == p_min_on_kw)
    ].copy()

    if "Converged" in df.columns:
        df = df[df["Converged"]].copy()

    df = df.dropna(subset=["Total_Loss_kWh", "Prod_Power_kWh"]).copy()

    if df.empty:
        raise ValueError(
            "No valid optimization results found for the selected plotting case."
        )

    # ============================================================
    # Section 1: segment comparison
    # ============================================================
    seg_df = df[
        (df["Operating_Case"] == base_operating_case_for_segment_section) &
        (df["Segments"].isin(segment_list))
    ].copy()

    if seg_df.empty:
        raise ValueError(
            f"No segment results found for operating case "
            f"{base_operating_case_for_segment_section}."
        )

    seg_df = seg_df.sort_values("Segments").copy()

    max_seg_loss = seg_df["Total_Loss_kWh"].max()

    if max_seg_loss <= 0:
        raise ValueError("Maximum segment loss is zero; cannot compute relative loss.")

    seg_df["Relative_Loss_pct"] = 100.0 * seg_df["Total_Loss_kWh"] / max_seg_loss
    seg_df["Absolute_Loss_MWh"] = seg_df["Total_Loss_kWh"] / 1000.0

    seg_df["Realized_Loss_pct"] = np.where(
        seg_df["Prod_Power_kWh"] > 1e-9,
        100.0 * seg_df["Total_Loss_kWh"] / seg_df["Prod_Power_kWh"],
        0.0
    )

    # ============================================================
    # Section 2: operating strategy comparison
    # ============================================================
    state_df = df[
        (df["Operating_Case"].isin(operating_cases)) &
        (df["Segments"].isin(state_segments))
    ].copy()

    if state_df.empty:
        raise ValueError("No operating-strategy comparison results found.")

    max_state_loss = state_df["Total_Loss_kWh"].max()

    if max_state_loss <= 0:
        raise ValueError("Maximum state loss is zero; cannot compute relative loss.")

    state_df["Relative_Loss_pct"] = 100.0 * state_df["Total_Loss_kWh"] / max_state_loss
    state_df["Absolute_Loss_MWh"] = state_df["Total_Loss_kWh"] / 1000.0

    state_df["Realized_Loss_pct"] = np.where(
        state_df["Prod_Power_kWh"] > 1e-9,
        100.0 * state_df["Total_Loss_kWh"] / state_df["Prod_Power_kWh"],
        0.0
    )

    # ============================================================
    # Academic style
    # ============================================================
    plt.rcParams.update(
        {
            "font.family": "serif",
            "font.size": 11,
            "axes.labelsize": 12,
            "axes.titlesize": 12,
            "xtick.labelsize": 10,
            "ytick.labelsize": 11,
            "legend.fontsize": 10,
            "axes.linewidth": 1.0,
            "hatch.linewidth": 0.8,
            "savefig.dpi": 600
        }
    )

    fig, ax1 = plt.subplots(figsize=(10.5, 4.9))
    ax2 = ax1.twinx()

    segment_x = np.arange(len(segment_list), dtype=float)

    gap = 1.55
    state_start = segment_x[-1] + gap + 1.0
    state_x = state_start + np.arange(len(operating_cases), dtype=float)

    bar_width = 0.36

    color_segment = "0.76"
    color_1seg = "0.84"
    color_12seg = "0.62"
    black_color = "0.00"

    hatch_segment = "\\\\\\"
    hatch_1seg = "////"
    hatch_12seg = "...."

    # ============================================================
    # Segment section
    # ============================================================
    seg_rel_values = []
    seg_abs_values = []
    seg_overlay_values = []

    for seg in segment_list:
        row = seg_df[seg_df["Segments"] == seg]

        if row.empty:
            seg_rel_values.append(np.nan)
            seg_abs_values.append(np.nan)
            seg_overlay_values.append(np.nan)
        else:
            rel_loss = float(row["Relative_Loss_pct"].iloc[0])
            abs_loss = float(row["Absolute_Loss_MWh"].iloc[0])
            realized_loss = float(row["Realized_Loss_pct"].iloc[0])

            seg_rel_values.append(rel_loss)
            seg_abs_values.append(abs_loss)
            seg_overlay_values.append(min(realized_loss, 2.0))

    ax1.bar(
        segment_x,
        seg_rel_values,
        width=bar_width,
        color=color_segment,
        edgecolor="black",
        linewidth=0.9,
        hatch=hatch_segment,
        label="Segment loss"
    )

    ax1.bar(
        segment_x,
        seg_overlay_values,
        width=bar_width,
        bottom=seg_rel_values,
        color=black_color,
        edgecolor="black",
        linewidth=0.6,
        label="Realized loss overlay"
    )

    ax2.plot(
        segment_x,
        seg_abs_values,
        color="black",
        linewidth=1.0,
        marker="o",
        markersize=3.8,
        linestyle="-",
        label="Absolute loss"
    )

    # ============================================================
    # Operating strategy section
    # ============================================================
    for seg in state_segments:

        if seg == state_segments[0]:
            offset = -bar_width / 2
            color = color_1seg
            hatch = hatch_1seg
            label = f"{seg}-segment"
            linestyle = "--"
        else:
            offset = bar_width / 2
            color = color_12seg
            hatch = hatch_12seg
            label = f"{seg}-segment"
            linestyle = ":"

        rel_values = []
        abs_values = []
        overlay_values = []

        for case in operating_cases:
            row = state_df[
                (state_df["Operating_Case"] == case) &
                (state_df["Segments"] == seg)
            ]

            if row.empty:
                rel_values.append(np.nan)
                abs_values.append(np.nan)
                overlay_values.append(np.nan)
            else:
                rel_loss = float(row["Relative_Loss_pct"].iloc[0])
                abs_loss = float(row["Absolute_Loss_MWh"].iloc[0])
                realized_loss = float(row["Realized_Loss_pct"].iloc[0])

                rel_values.append(rel_loss)
                abs_values.append(abs_loss)
                overlay_values.append(min(realized_loss, 2.0))

        x_positions = state_x + offset

        ax1.bar(
            x_positions,
            rel_values,
            width=bar_width,
            color=color,
            edgecolor="black",
            linewidth=0.9,
            hatch=hatch,
            label=label
        )

        ax1.bar(
            x_positions,
            overlay_values,
            width=bar_width,
            bottom=rel_values,
            color=black_color,
            edgecolor="black",
            linewidth=0.6
        )

        ax2.plot(
            x_positions,
            abs_values,
            color="black",
            linewidth=1.0,
            marker="o",
            markersize=3.8,
            linestyle=linestyle
        )

    # ============================================================
    # Axis formatting
    # ============================================================
    all_xticks = list(segment_x) + list(state_x)
    all_xtick_labels = (
        [str(s) for s in segment_list]
        + [OPERATING_CASE_LABELS[c] for c in operating_cases]
    )

    ax1.set_xticks(all_xticks)
    ax1.set_xticklabels(all_xtick_labels)

    ax1.set_ylabel("Relative loss (%)")
    ax2.set_ylabel("Absolute loss (MWh)")
    ax1.set_xlabel("Number of segments                                      Operating strategy")

    ax1.set_ylim(0, 105)

    abs_max = max(
        np.nanmax(seg_abs_values) if len(seg_abs_values) > 0 and not np.all(np.isnan(seg_abs_values)) else 0.0,
        state_df["Absolute_Loss_MWh"].max()
    )

    ax2.set_ylim(0, abs_max * 1.12 if abs_max > 0 else 1.0)

    divider_x = segment_x[-1] + gap / 2

    ax1.axvline(
        divider_x,
        color="0.2",
        linestyle=":",
        linewidth=1.0
    )

    ax1.grid(
        True,
        axis="y",
        linestyle="-",
        linewidth=0.55,
        color="0.82"
    )

    ax1.set_axisbelow(True)

    for spine in ax1.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)

    for spine in ax2.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)

    # ============================================================
    # Legend
    # ============================================================
    legend_handles = [
        Patch(
            facecolor=color_segment,
            edgecolor="black",
            hatch=hatch_segment,
            label="Segment section: on/off/standby"
        ),
        Patch(
            facecolor=color_1seg,
            edgecolor="black",
            hatch=hatch_1seg,
            label="1-segment operating strategy"
        ),
        Patch(
            facecolor=color_12seg,
            edgecolor="black",
            hatch=hatch_12seg,
            label="12-segment operating strategy"
        ),
        Patch(
            facecolor=black_color,
            edgecolor="black",
            label="Realized loss overlay"
        ),
        Line2D(
            [0],
            [0],
            color="black",
            marker="o",
            linewidth=1.0,
            label="Absolute loss, right axis"
        )
    ]

    ax1.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.24),
        ncol=2,
        frameon=True,
        fancybox=False,
        edgecolor="black",
        framealpha=1.0
    )

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.30)

    fig.savefig(output_png, bbox_inches="tight")
    fig.savefig(output_pdf, bbox_inches="tight")

    plt.show()

    print("\nSaved loss comparison plot:")
    print(output_png)
    print(output_pdf)


# ============================================================
# Load electrolyzer efficiency/loss curve data
# ============================================================
data = pd.read_csv(EFFICIENCY_FILE)

data["Power"] = data["Power_kW_per_m2"] * 100

temps = sorted(data["Temperature_C"].unique())

if SELECTED_TEMPS is None:
    selected_temps = temps
else:
    selected_temps = SELECTED_TEMPS


# ============================================================
# Load wind time series
# ============================================================
wind_df = read_renewables_ninja_csv(WIND_FILE)

if "electricity" not in wind_df.columns:
    raise ValueError(
        f"Wind file columns are {list(wind_df.columns)}; expected an 'electricity' column."
    )

if "time" not in wind_df.columns:
    raise ValueError(
        f"Wind file columns are {list(wind_df.columns)}; expected a 'time' column."
    )

wind_df["datetime"] = pd.to_datetime(wind_df["time"], utc=True, errors="coerce")
wind_df["electricity"] = pd.to_numeric(wind_df["electricity"], errors="coerce")

wind_df = wind_df.dropna(subset=["datetime", "electricity"]).copy()

if SCALE_NINJA_TO_CAPACITY:
    if wind_df["electricity"].max() <= 0:
        raise ValueError("Ninja electricity column has no positive values.")

    wind_df["P_RE_kW"] = (
        wind_df["electricity"] / wind_df["electricity"].max()
    ) * P_RE_CAPACITY_KW
else:
    wind_df["P_RE_kW"] = wind_df["electricity"]

wind_df = wind_df[["datetime", "P_RE_kW"]].sort_values("datetime").reset_index(drop=True)


# ============================================================
# Load electricity price
# ============================================================
price_df = read_energy_charts_csv(ENERGY_CHARTS_FILE)

date_candidates = [c for c in price_df.columns if "Date" in c]
if not date_candidates:
    raise ValueError(f"Could not find Date column. Columns are: {list(price_df.columns)}")

date_col = date_candidates[0]

target_price_name = f"Day Ahead Auction ({PRICE_ZONE})"
price_candidates = [c for c in price_df.columns if target_price_name in c]

if not price_candidates:
    raise ValueError(
        f"Could not find price column containing '{target_price_name}'. "
        f"Columns are: {list(price_df.columns)}"
    )

price_col = price_candidates[0]

price_df["datetime"] = pd.to_datetime(price_df[date_col], utc=True, errors="coerce")
price_df["price_EUR_MWh"] = pd.to_numeric(price_df[price_col], errors="coerce")

price_df = price_df.dropna(subset=["datetime", "price_EUR_MWh"]).copy()

price_df["price_buy"] = price_df["price_EUR_MWh"] / 1000.0

price_df = price_df[["datetime", "price_buy"]].sort_values("datetime").reset_index(drop=True)


# ============================================================
# Merge wind and price
# ============================================================
time_data = pd.merge_asof(
    wind_df.sort_values("datetime"),
    price_df.sort_values("datetime"),
    on="datetime",
    direction="nearest",
    tolerance=pd.Timedelta("90min")
)

time_data = time_data.dropna(subset=["P_RE_kW", "price_buy"]).copy()
time_data = time_data.sort_values("datetime").reset_index(drop=True)

if time_data.empty:
    raise ValueError(
        "Merged time_data is empty. Check time zones or timestamps in Ninja and Energy-Charts files."
    )

if HORIZON_HOURS is not None:
    time_data = time_data.iloc[:HORIZON_HOURS].copy()

time_data["price_sell"] = time_data["price_buy"]
time_data["demand_kg"] = 7

T = len(time_data)
p_re = time_data["P_RE_kW"].tolist()
price_buy = time_data["price_buy"].tolist()
price_sell = time_data["price_sell"].tolist()
E_d_series = time_data["demand_kg"].tolist()

time_data.to_csv(MERGED_TIME_FILE, index=False)

print("\nMerged time-series data:")
print(time_data.head())
print(time_data.tail())
print("\nRows used:", len(time_data))
print("P_RE range kW:", time_data["P_RE_kW"].min(), "to", time_data["P_RE_kW"].max())
print("Price range EUR/kWh:", time_data["price_buy"].min(), "to", time_data["price_buy"].max())
print("Merged file saved to:", MERGED_TIME_FILE)


# ============================================================
# Fixed model parameters
# ============================================================
P_el_max = 1500

e_min = 0
e_max = 2000
e_0 = 300

C_hyo = 5
HHV = 39.39
dt = 1

P_stb = 25
C_start = 1500

M_grid = max(P_el_max, max(p_re))


# ============================================================
# Results containers
# ============================================================
sensitivity_results = []
all_hourly_results = []
computational_results = []


# ============================================================
# Optimization loop
# ============================================================
for operating_case in OPERATING_CASES:

    operating_label = OPERATING_CASE_LABELS[operating_case]

    print(f"\nSolving operating strategy: {operating_label}")

    for P_min_on in P_MIN_VALUES:

        print(f"\nSolving model with P_min_on = {P_min_on} kW")

        for temp in selected_temps:

            data_temp = data[data["Temperature_C"] == temp].copy()

            if data_temp.empty:
                raise ValueError(
                    f"Temperature {temp} not found. Available temperatures: {temps}"
                )

            data_temp["loss_curve"] = (
                1 - data_temp["Efficiency_eta"]
            ) * data_temp["Power"]

            for n_segments in CASE_SEGMENT_OPTIONS[operating_case]:

                p_breaks = list(np.linspace(0.0, P_el_max, n_segments + 1))

                df_fit = (
                    data_temp[["Power", "loss_curve"]]
                    .sort_values("Power")
                    .drop_duplicates("Power")
                    .copy()
                )

                if df_fit["Power"].min() > 0:
                    df_fit = pd.concat(
                        [
                            pd.DataFrame(
                                {
                                    "Power": [0.0],
                                    "loss_curve": [0.0]
                                }
                            ),
                            df_fit
                        ],
                        ignore_index=True
                    ).sort_values("Power")

                loss_breaks = np.interp(
                    p_breaks,
                    df_fit["Power"],
                    df_fit["loss_curve"]
                )

                s = []

                for i in range(len(p_breaks) - 1):
                    dx = p_breaks[i + 1] - p_breaks[i]
                    dy = loss_breaks[i + 1] - loss_breaks[i]

                    if abs(dx) < 1e-9:
                        s.append(0.0)
                    else:
                        s.append(dy / dx)

                negative_slopes = [val for val in s if val < -1e-9]

                if negative_slopes:
                    print(
                        f"Warning: negative loss slopes at T={temp}, "
                        f"operating_case={operating_case}, "
                        f"segments={n_segments}: {negative_slopes}"
                    )

                delta_p = [
                    p_breaks[i + 1] - p_breaks[i]
                    for i in range(len(p_breaks) - 1)
                ]

                N_seg = len(s)

                m = gp.Model(
                    f"electrolyzer_{operating_case}_T{temp}_S{n_segments}_Pmin{P_min_on}"
                )

                m.setParam("OutputFlag", 0)
                m.setParam("TimeLimit", 600)
                m.setParam("MIPGap", 0.005)
                m.setParam("MIPFocus", 1)
                m.setParam("Threads", 0)

                # Decision variables
                p_el = m.addVars(T, lb=0, ub=P_el_max, name="p_el")
                p_prod = m.addVars(T, lb=0, ub=P_el_max, name="p_prod")
                p_grid_buy = m.addVars(T, lb=0, name="p_grid_buy")
                p_sell = m.addVars(T, lb=0, name="p_sell")

                e = m.addVars(T, lb=e_min, ub=e_max, name="e")
                e_el = m.addVars(T, lb=0, name="e_el")
                loss = m.addVars(T, lb=0, name="loss")

                p_ti = m.addVars(T, N_seg, lb=0, name="p_ti")
                z = m.addVars(T, N_seg, vtype=GRB.BINARY, name="z")

                y_on = m.addVars(T, vtype=GRB.BINARY, name="y_on")
                y_stb = m.addVars(T, vtype=GRB.BINARY, name="y_stb")
                y_buy = m.addVars(T, vtype=GRB.BINARY, name="y_buy")
                y_start = m.addVars(T, vtype=GRB.BINARY, name="y_start")

                # Objective
                m.setObjective(
                    gp.quicksum(C_hyo * e_el[t] for t in range(T))
                    + gp.quicksum(
                        price_sell[t] * p_sell[t] * dt
                        for t in range(T)
                    )
                    - gp.quicksum(
                        price_buy[t] * p_grid_buy[t] * dt
                        for t in range(T)
                    )
                    - gp.quicksum(
                        C_start * y_start[t]
                        for t in range(T)
                    ),
                    GRB.MAXIMIZE
                )

                # Constraints
                for t in range(T):

                    m.addConstr(
                        loss[t] ==
                        gp.quicksum(
                            s[i] * p_ti[t, i]
                            for i in range(N_seg)
                        ),
                        name=f"loss_pwl_{t}"
                    )

                    for i in range(N_seg):
                        m.addConstr(
                            p_ti[t, i] <= delta_p[i] * z[t, i],
                            name=f"seg_bound_{t}_{i}"
                        )

                    m.addConstr(
                        p_prod[t] ==
                        gp.quicksum(
                            p_ti[t, i]
                            for i in range(N_seg)
                        ),
                        name=f"prod_decomp_{t}"
                    )

                    for i in range(1, N_seg):
                        m.addConstr(
                            z[t, i] <= z[t, i - 1],
                            name=f"seq_seg_{t}_{i}"
                        )

                        m.addConstr(
                            p_ti[t, i - 1] >=
                            delta_p[i - 1] * z[t, i],
                            name=f"seg_contiguity_{t}_{i}"
                        )

                    m.addConstr(
                        y_on[t] + y_stb[t] <= 1,
                        name=f"state_excl_{t}"
                    )

                    m.addConstr(
                        p_prod[t] <= P_el_max * y_on[t],
                        name=f"prod_max_{t}"
                    )

                    m.addConstr(
                        p_prod[t] >= P_min_on * y_on[t],
                        name=f"prod_min_{t}"
                    )

                    m.addConstr(
                        p_el[t] ==
                        p_prod[t] + P_stb * y_stb[t],
                        name=f"p_el_def_{t}"
                    )

                    m.addConstr(
                        e_el[t] ==
                        (p_prod[t] - loss[t]) * dt / HHV,
                        name=f"h2_prod_{t}"
                    )

                    m.addConstr(
                        loss[t] <= p_prod[t],
                        name=f"loss_bound_{t}"
                    )
                   # p_curt = m.addVars(T, lb=0, name="p_curt")
                    m.addConstr(
                        p_grid_buy[t] + p_re[t] ==
                        p_el[t] + p_sell[t],
                        name=f"power_balance_{t}"
                    )

                    m.addConstr(
                        p_grid_buy[t] <= M_grid * y_buy[t],
                        name=f"buy_mode_{t}"
                    )

                    m.addConstr(
                        p_sell[t] <= M_grid * (1 - y_buy[t]),
                        name=f"sell_mode_{t}"
                    )

                    if t == 0:
                        m.addConstr(
                            e[0] ==
                            e_0 + e_el[0] - E_d_series[0],
                            name="storage_0"
                        )
                    else:
                        m.addConstr(
                            e[t] ==
                            e[t - 1] + e_el[t] - E_d_series[t],
                            name=f"storage_{t}"
                        )

                    add_operating_case_constraints(
                        m=m,
                        t=t,
                        case_name=operating_case,
                        y_on=y_on,
                        y_stb=y_stb
                    )

                    if t == 0:
                        m.addConstr(
                            y_start[0] == y_on[0],
                            name="startup_initial"
                        )
                    else:
                        m.addConstr(
                            y_start[t] >=
                            y_on[t] -
                            y_on[t - 1] -
                            y_stb[t - 1],
                            name=f"startup_lb_{t}"
                        )

                        m.addConstr(
                            y_start[t] <= y_on[t],
                            name=f"startup_ub_on_{t}"
                        )

                        m.addConstr(
                            y_start[t] <= 1 - y_on[t - 1],
                            name=f"startup_ub_prev_on_{t}"
                        )

                        m.addConstr(
                            y_start[t] <= 1 - y_stb[t - 1],
                            name=f"startup_ub_prev_stb_{t}"
                        )

                # Terminal storage fairness constraint:
                # This prevents artificial profit from draining the tank at the horizon end.
                m.addConstr(e[T - 1] >= e_0, name="terminal_storage_min")

                m.optimize()

                runtime_seconds = float(m.Runtime)
                binary_count = count_binary_variables(m)

                computational_results.append(
                    {
                        "Case": f"{operating_label}-{n_segments}",
                        "Operating_Case": operating_case,
                        "Operating_Label": operating_label,
                        "Segments": n_segments,
                        "Temperature_C": temp,
                        "P_min_on_kW": P_min_on,
                        "Horizon_hours": T,
                        "Computational_time_s": runtime_seconds,
                        "Binary_variables": binary_count,
                        "Binary_variables_formatted": format_binary_count(binary_count, T),
                        "Gurobi_Status": m.status,
                        "MIPGap": m.MIPGap if m.SolCount > 0 else None,
                        "Converged": bool(m.status == GRB.OPTIMAL or (m.SolCount > 0 and m.MIPGap <= 0.01))
                    }
                )

                if m.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
                    has_solution = m.SolCount > 0
                else:
                    has_solution = False

                if not has_solution:
                    print(
                        f"  WARNING: NO feasible solution for {operating_label}-{n_segments} "
                        f"(status {m.status})"
                    )
                elif m.MIPGap > 0.01:
                    print(
                        f"  WARNING: {operating_label}-{n_segments} stopped at gap "
                        f"{m.MIPGap:.2%} — unreliable for plotting/paper results"
                    )

                if has_solution:

                    total_h2 = sum(e_el[t].X for t in range(T))
                    total_loss = sum(loss[t].X * dt for t in range(T))
                    total_grid_buy = sum(p_grid_buy[t].X * dt for t in range(T))
                    total_sell = sum(p_sell[t].X * dt for t in range(T))
                    total_power = sum(p_el[t].X * dt for t in range(T))
                    total_prod_power = sum(p_prod[t].X * dt for t in range(T))

                    standby_hours = sum(y_stb[t].X for t in range(T))
                    on_hours = sum(y_on[t].X for t in range(T))
                    off_hours = T - standby_hours - on_hours

                    total_startups = sum(y_start[t].X for t in range(T))

                    total_buy_cost = sum(
                        price_buy[t] * p_grid_buy[t].X * dt
                        for t in range(T)
                    )

                    total_sell_rev = sum(
                        price_sell[t] * p_sell[t].X * dt
                        for t in range(T)
                    )

                    total_start_cost = C_start * total_startups

                    sensitivity_results.append(
                        {
                            "Operating_Case": operating_case,
                            "Operating_Label": operating_label,
                            "Strategy": operating_label,
                            "P_min_on_kW": P_min_on,
                            "Temperature_C": temp,
                            "Fixed_Temp_C": temp,
                            "Segments": n_segments,
                            "Horizon_hours": T,
                            "Status": m.status,
                            "MIPGap": m.MIPGap if m.SolCount > 0 else None,
                            "Converged": bool(m.status == GRB.OPTIMAL or (m.SolCount > 0 and m.MIPGap <= 0.01)),
                            "Profit": m.objVal,
                            "Total_H2_kg": total_h2,
                            "Total_Loss_kWh": total_loss,
                            "Total_Loss_MWh": total_loss / 1000.0,
                            "Grid_Buy_Energy_kWh": total_grid_buy,
                            "Curtailment_kWh": 0.0,
                            "Sell_Energy_kWh": total_sell,
                            "Buy_Cost_EUR": total_buy_cost,
                            "Sell_Revenue_EUR": total_sell_rev,
                            "Startup_Cost": total_start_cost,
                            "Total_Startups": total_startups,
                            "Total_Power_kWh": total_power,
                            "Prod_Power_kWh": total_prod_power,
                            "Final_Storage_kg": e[T - 1].X,
                            "Standby_Hours": standby_hours,
                            "On_Hours": on_hours,
                            "Off_Hours": off_hours,
                            "Mean_P_RE_kW": float(np.mean(p_re)),
                            "Max_P_RE_kW": float(np.max(p_re)),
                            "Mean_price_buy_EUR_kWh": float(np.mean(price_buy))
                        }
                    )

                    for t in range(T):
                        all_hourly_results.append(
                            {
                                "Operating_Case": operating_case,
                                "Operating_Label": operating_label,
                                "P_min_on_kW": P_min_on,
                                "Temperature_C": temp,
                                "Segments": n_segments,
                                "Time": t,
                                "DateTime": time_data.iloc[t]["datetime"],
                                "Price_Buy_EUR_kWh": price_buy[t],
                                "Price_Sell_EUR_kWh": price_sell[t],
                                "Demand_kg": E_d_series[t],
                                "Loss_t_kW": loss[t].X,
                                "P_el_t_kW": p_el[t].X,
                                "P_prod_t_kW": p_prod[t].X,
                                "P_grid_buy_t_kW": p_grid_buy[t].X,
                                "P_sell_t_kW": p_sell[t].X,
                                "P_renewable_t_kW": p_re[t],
                                "H2_t_kg": e_el[t].X,
                                "Storage_t_kg": e[t].X,
                                "State_On": y_on[t].X,
                                "State_Standby": y_stb[t].X,
                                "Buy_Mode": y_buy[t].X,
                                "Startup": y_start[t].X
                            }
                        )

                    print(
                        f"Done: case={operating_label}, "
                        f"segments={n_segments}, "
                        f"loss={total_loss / 1000.0:.3f} MWh, "
                        f"runtime={runtime_seconds:.1f} s, "
                        f"binary={format_binary_count(binary_count, T)}, "
                        f"profit={m.objVal:.2f}"
                    )

                else:
                    print(
                        f"No feasible solution: case={operating_label}, "
                        f"segments={n_segments}, "
                        f"status={m.status}, "
                        f"runtime={runtime_seconds:.1f} s"
                    )



# ============================================================
# Convergence diagnostics
# ============================================================
def print_convergence_summary(sensitivity_df: pd.DataFrame, computational_df: pd.DataFrame):
    print("\nCONVERGENCE SUMMARY")

    if computational_df.empty:
        print("No computational results available.")
        return

    cols = [
        "Case",
        "Operating_Label",
        "Segments",
        "Temperature_C",
        "P_min_on_kW",
        "Gurobi_Status",
        "MIPGap",
        "Converged",
        "Computational_time_s",
        "Binary_variables_formatted"
    ]

    cols = [c for c in cols if c in computational_df.columns]
    print(computational_df[cols].to_string(index=False))

    if not sensitivity_df.empty and "Converged" in sensitivity_df.columns:
        bad = sensitivity_df[~sensitivity_df["Converged"]].copy()
        if bad.empty:
            print("\nAll sensitivity cases used for plots are converged.")
        else:
            print("\nNon-converged cases excluded from plots:")
            print(
                bad[
                    [
                        "Operating_Label",
                        "Segments",
                        "Temperature_C",
                        "P_min_on_kW",
                        "Status",
                        "MIPGap",
                        "Profit"
                    ]
                ].to_string(index=False)
            )


# ============================================================
# Create final DataFrames
# ============================================================
sensitivity_df = pd.DataFrame(sensitivity_results)
hourly_sensitivity_df = pd.DataFrame(all_hourly_results)
computational_df = pd.DataFrame(computational_results)

print("\nOptimization results:")
print(sensitivity_df)

print("\nComputational results:")
print(computational_df)

print_convergence_summary(sensitivity_df, computational_df)



# ============================================================
# Save results
# ============================================================
sensitivity_df.to_csv(sensitivity_out, index=False)
hourly_sensitivity_df.to_csv(hourly_out, index=False)
computational_df.to_csv(computational_out, index=False)

print("\nSaved:")
print(sensitivity_out)
print(hourly_out)
print(computational_out)
print(MERGED_TIME_FILE)


# ============================================================
# Table II: fixed-temperature sensitivity
# ON/OFF/STANDBY, 12 segments only
# ============================================================
temperature_table_df = sensitivity_df[
    (sensitivity_df["Operating_Case"] == "ON_OFF_STANDBY") &
    (sensitivity_df["Segments"] == 12)
].copy()

if not temperature_table_df.empty:
    if "Fixed_Temp_C" not in temperature_table_df.columns:
        temperature_table_df["Fixed_Temp_C"] = temperature_table_df["Temperature_C"]

    if "Strategy" not in temperature_table_df.columns:
        temperature_table_df["Strategy"] = temperature_table_df["Operating_Label"]

    if "Curtailment_kWh" not in temperature_table_df.columns:
        # The uploaded model has no curtailment variable and uses hard equality
        # p_grid_buy + p_re = p_el + p_sell, so modeled curtailment is zero.
        temperature_table_df["Curtailment_kWh"] = 0.0

    runtime_df = computational_df[
        [
            "Temperature_C",
            "Operating_Case",
            "Segments",
            "Computational_time_s"
        ]
    ].copy()

    temperature_table_df = temperature_table_df.merge(
        runtime_df,
        on=["Temperature_C", "Operating_Case", "Segments"],
        how="left"
    )

    temperature_table_df = temperature_table_df.rename(
        columns={
            "Profit": "Profit_EUR",
            "Total_H2_kg": "H2_produced_kg",
            "Total_Loss_kWh": "Total_loss_kWh",
            "Grid_Buy_Energy_kWh": "Grid_import_kWh",
            "MIPGap": "MIP_gap",
            "Computational_time_s": "Runtime_s",
        }
    )

    temperature_table_df = temperature_table_df[
        [
            "Fixed_Temp_C",
            "Temperature_C",
            "Strategy",
            "Segments",
            "Profit_EUR",
            "H2_produced_kg",
            "Total_loss_kWh",
            "Grid_import_kWh",
            "Curtailment_kWh",
            "MIP_gap",
            "Runtime_s",
            "Converged",
            "Horizon_hours",
        ]
    ].copy()

    temperature_table_df = temperature_table_df.sort_values("Fixed_Temp_C").reset_index(drop=True)

    temperature_table_df.to_csv(temperature_table_out, index=False)

    print("\nTABLE II: TEMPERATURE SENSITIVITY")
    print("Fixed-temperature sensitivity: ON/OFF/STANDBY, 12 segments\n")
    print(
        temperature_table_df[
            [
                "Fixed_Temp_C",
                "Profit_EUR",
                "H2_produced_kg",
                "Total_loss_kWh",
                "Grid_import_kWh",
                "Curtailment_kWh",
                "MIP_gap",
                "Runtime_s",
            ]
        ].to_string(index=False)
    )

    print("\nTemperature-sensitivity Table II CSV saved to:")
    print(temperature_table_out)
else:
    print("\nNo ON/OFF/STANDBY 12-segment temperature-sensitivity results were found.")


# ============================================================
# Print computational table
# ============================================================
computational_table_df = print_computational_aspects_table(
    computational_df=computational_df,
    temperature_c=85,
    p_min_on_kw=PLOT_P_MIN_ON,
    case_order=[
        "on/off/standby-12"
    ],
    output_csv=computational_out,
    output_latex=computational_latex_out
)

# ============================================================
# Final profit-comparison plot
# ============================================================
def plot_profit_comparison_on_off_standby(
    sensitivity_df: pd.DataFrame,
    output_png: str,
    output_pdf: str,
    temperature_c: float = 85,
    p_min_on_kw: float = 450,
    segment_list=(1, 2, 4, 8, 12),
    operating_cases=("ON_OFF", "ON_OFF_STANDBY"),
    state_segments=(1, 12),
    base_operating_case_for_segment_section="ON_OFF_STANDBY"
):
    """
    Creates the final profit comparison figure.

    X-axis:
        Section 1: number of segments, 1, 2, 4, 8, 12
        Section 2: operating strategies, on/off and on/off/standby

    Left y-axis:
        Relative profit (%)

    Right y-axis:
        Absolute profit (kEUR)

    Bars:
        Relative profit.

    Markers:
        Absolute profit.
    """

    df = sensitivity_df.copy()

    df = df[
        (df["Temperature_C"] == temperature_c) &
        (df["P_min_on_kW"] == p_min_on_kw)
    ].copy()

    if "Converged" in df.columns:
        df = df[df["Converged"]].copy()

    df = df.dropna(subset=["Profit"]).copy()

    if df.empty:
        raise ValueError(
            "No valid optimization results found for the selected plotting case."
        )

    # ============================================================
    # Section 1: segment comparison
    # ============================================================
    seg_df = df[
        (df["Operating_Case"] == base_operating_case_for_segment_section) &
        (df["Segments"].isin(segment_list))
    ].copy()

    if seg_df.empty:
        raise ValueError(
            f"No segment results found for operating case "
            f"{base_operating_case_for_segment_section}."
        )

    seg_df = seg_df.sort_values("Segments").copy()

    seg_profit_scale = seg_df["Profit"].abs().max()

    if seg_profit_scale <= 0:
        raise ValueError("Maximum segment profit magnitude is zero; cannot compute relative profit.")

    seg_df["Relative_Profit_pct"] = 100.0 * seg_df["Profit"] / seg_profit_scale
    seg_df["Absolute_Profit_kEUR"] = seg_df["Profit"] / 1000.0

    # ============================================================
    # Section 2: operating strategy comparison
    # ============================================================
    state_df = df[
        (df["Operating_Case"].isin(operating_cases)) &
        (df["Segments"].isin(state_segments))
    ].copy()

    if state_df.empty:
        raise ValueError("No operating-strategy comparison results found.")

    state_profit_scale = state_df["Profit"].abs().max()

    if state_profit_scale <= 0:
        raise ValueError("Maximum state profit magnitude is zero; cannot compute relative profit.")

    state_df["Relative_Profit_pct"] = 100.0 * state_df["Profit"] / state_profit_scale
    state_df["Absolute_Profit_kEUR"] = state_df["Profit"] / 1000.0

    # ============================================================
    # Academic style
    # ============================================================
    plt.rcParams.update(
        {
            "font.family": "serif",
            "font.size": 11,
            "axes.labelsize": 12,
            "axes.titlesize": 12,
            "xtick.labelsize": 10,
            "ytick.labelsize": 11,
            "legend.fontsize": 10,
            "axes.linewidth": 1.0,
            "hatch.linewidth": 0.8,
            "savefig.dpi": 600
        }
    )

    fig, ax1 = plt.subplots(figsize=(10.5, 4.9))
    ax2 = ax1.twinx()

    segment_x = np.arange(len(segment_list), dtype=float)

    gap = 1.55
    state_start = segment_x[-1] + gap + 1.0
    state_x = state_start + np.arange(len(operating_cases), dtype=float)

    bar_width = 0.36

    color_segment = "0.76"
    color_1seg = "0.84"
    color_12seg = "0.62"

    hatch_segment = "\\\\\\"
    hatch_1seg = "////"
    hatch_12seg = "...."

    # ============================================================
    # Segment section
    # ============================================================
    seg_rel_values = []
    seg_abs_values = []

    for seg in segment_list:
        row = seg_df[seg_df["Segments"] == seg]

        if row.empty:
            seg_rel_values.append(np.nan)
            seg_abs_values.append(np.nan)
        else:
            seg_rel_values.append(float(row["Relative_Profit_pct"].iloc[0]))
            seg_abs_values.append(float(row["Absolute_Profit_kEUR"].iloc[0]))

    ax1.bar(
        segment_x,
        seg_rel_values,
        width=bar_width,
        color=color_segment,
        edgecolor="black",
        linewidth=0.9,
        hatch=hatch_segment,
        label="Segment profit"
    )

    ax2.plot(
        segment_x,
        seg_abs_values,
        color="black",
        linewidth=1.0,
        marker="o",
        markersize=3.8,
        linestyle="-",
        label="Absolute profit"
    )

    # ============================================================
    # Operating strategy section
    # ============================================================
    for seg in state_segments:

        if seg == state_segments[0]:
            offset = -bar_width / 2
            color = color_1seg
            hatch = hatch_1seg
            label = f"{seg}-segment"
            linestyle = "--"
        else:
            offset = bar_width / 2
            color = color_12seg
            hatch = hatch_12seg
            label = f"{seg}-segment"
            linestyle = ":"

        rel_values = []
        abs_values = []

        for case in operating_cases:
            row = state_df[
                (state_df["Operating_Case"] == case) &
                (state_df["Segments"] == seg)
            ]

            if row.empty:
                rel_values.append(np.nan)
                abs_values.append(np.nan)
            else:
                rel_values.append(float(row["Relative_Profit_pct"].iloc[0]))
                abs_values.append(float(row["Absolute_Profit_kEUR"].iloc[0]))

        x_positions = state_x + offset

        ax1.bar(
            x_positions,
            rel_values,
            width=bar_width,
            color=color,
            edgecolor="black",
            linewidth=0.9,
            hatch=hatch,
            label=label
        )

        ax2.plot(
            x_positions,
            abs_values,
            color="black",
            linewidth=1.0,
            marker="o",
            markersize=3.8,
            linestyle=linestyle
        )

    # ============================================================
    # Axis formatting
    # ============================================================
    all_xticks = list(segment_x) + list(state_x)
    all_xtick_labels = (
        [str(s) for s in segment_list]
        + [OPERATING_CASE_LABELS[c] for c in operating_cases]
    )

    ax1.set_xticks(all_xticks)
    ax1.set_xticklabels(all_xtick_labels)

    ax1.set_ylabel("Relative profit (%)")
    ax2.set_ylabel("Absolute profit (kEUR)")
    ax1.set_xlabel("Number of segments                                      Operating strategy")

    # Allow negative profit if it occurs
    all_relative_values = np.array(seg_rel_values + list(state_df["Relative_Profit_pct"].values), dtype=float)
    all_relative_values = all_relative_values[~np.isnan(all_relative_values)]

    rel_min = float(np.min(all_relative_values))
    rel_max = float(np.max(all_relative_values))

    if rel_min >= 0:
        ax1.set_ylim(0, max(105, rel_max * 1.10))
    else:
        ax1.set_ylim(rel_min * 1.15, rel_max * 1.15)

    all_abs_values = np.array(seg_abs_values + list(state_df["Absolute_Profit_kEUR"].values), dtype=float)
    all_abs_values = all_abs_values[~np.isnan(all_abs_values)]

    abs_min = float(np.min(all_abs_values))
    abs_max = float(np.max(all_abs_values))

    if abs_min >= 0:
        ax2.set_ylim(0, abs_max * 1.12 if abs_max > 0 else 1.0)
    else:
        ax2.set_ylim(abs_min * 1.15, abs_max * 1.15)

    divider_x = segment_x[-1] + gap / 2

    ax1.axvline(
        divider_x,
        color="0.2",
        linestyle=":",
        linewidth=1.0
    )

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.grid(
        True,
        axis="y",
        linestyle="-",
        linewidth=0.55,
        color="0.82"
    )

    ax1.set_axisbelow(True)

    for spine in ax1.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)

    for spine in ax2.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)

    # ============================================================
    # Legend
    # ============================================================
    legend_handles = [
        Patch(
            facecolor=color_segment,
            edgecolor="black",
            hatch=hatch_segment,
            label="Segment section: on/off/standby"
        ),
        Patch(
            facecolor=color_1seg,
            edgecolor="black",
            hatch=hatch_1seg,
            label="1-segment operating strategy"
        ),
        Patch(
            facecolor=color_12seg,
            edgecolor="black",
            hatch=hatch_12seg,
            label="12-segment operating strategy"
        ),
        Line2D(
            [0],
            [0],
            color="black",
            marker="o",
            linewidth=1.0,
            label="Absolute profit, right axis"
        )
    ]

    ax1.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.24),
        ncol=2,
        frameon=True,
        fancybox=False,
        edgecolor="black",
        framealpha=1.0
    )

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.30)

    fig.savefig(output_png, bbox_inches="tight")
    fig.savefig(output_pdf, bbox_inches="tight")

    plt.show()

    print("\nSaved profit comparison plot:")
    print(output_png)
    print(output_pdf)

# ============================================================
# Generate final plots
# ============================================================
# Disabled for the temperature-sensitivity Table II run.
# This run intentionally solves only ON/OFF/STANDBY with 12 segments.
print("Automatic figure generation skipped for the temperature-sensitivity Table II run.")



# ============================================================
# Separate computational-aspects comparison:
# OO vs OOS at fixed 85 C, 8760 h, base case
# ============================================================
computational_OO_OOS_out = f"{BASE_DIR}/computational_aspects_OO_OOS.csv"
computational_OO_OOS_latex_out = f"{BASE_DIR}/computational_aspects_OO_OOS.tex"


def _build_computational_table_OO_OOS(
    computational_df: pd.DataFrame,
    output_csv: str,
    output_latex: str
) -> pd.DataFrame:
    case_order = ["OO-1", "OO-12", "OOS-1", "OOS-4", "OOS-8", "OOS-12"]

    table_df = computational_df.copy()
    table_df["Case_order"] = table_df["Case"].map(
        {case: i for i, case in enumerate(case_order)}
    )
    table_df = table_df.sort_values("Case_order").drop(columns=["Case_order"])

    table_df = table_df[
        [
            "Case",
            "Operating_Case",
            "Segments",
            "Temperature_C",
            "Horizon_hours",
            "Computational_time_s",
            "Binary_variables",
            "Binary_variables_formatted",
            "Gurobi_Status",
            "MIPGap",
            "Converged",
            "Profit_EUR",
        ]
    ].copy()

    table_df.to_csv(output_csv, index=False)

    latex_df = table_df[
        [
            "Case",
            "Computational_time_s",
            "Binary_variables_formatted",
            "MIPGap",
        ]
    ].copy()

    latex_df.rename(
        columns={
            "Computational_time_s": "Computational time [s]",
            "Binary_variables_formatted": "No. of binary variables",
            "MIPGap": "MIP gap",
        },
        inplace=True
    )

    latex_df["Computational time [s]"] = latex_df["Computational time [s]"].map(
        lambda x: f"{x:,.1f}" if pd.notna(x) else ""
    )

    latex_df["MIP gap"] = latex_df["MIP gap"].map(
        lambda x: f"{x:.4f}" if pd.notna(x) else ""
    )

    latex_text = latex_df.to_latex(
        index=False,
        escape=False,
        column_format="lccc",
        caption="Computational aspects of the on/off and on/off/standby models.",
        label="tab:computational_aspects_oo_oos"
    )

    with open(output_latex, "w", encoding="utf-8") as f:
        f.write(latex_text)

    print("\nCOMPUTATIONAL ASPECTS: OO vs OOS")
    print(latex_df.to_string(index=False))
    print("\nSaved computational-aspects CSV:")
    print(output_csv)
    print("Saved computational-aspects LaTeX:")
    print(output_latex)

    return table_df


def run_computational_aspects_OO_OOS():
    """
    Focused computational-aspects run for the paper table.

    This run is separate from the temperature-sensitivity run above.

    Fixed case:
        Temperature_C = 85
        Horizon = 8760 h
        P_min_on = 450 kW
        P_stb = 25 kW
        C_start = 1500 EUR/start
        demand = 7 kg/h
        price_sell = price_buy
        TimeLimit = 600 s
        MIPGap = 0.005

    Compared cases:
        OO-1, OO-12, OOS-1, OOS-4, OOS-8, OOS-12
    """

    local_temperature = 85
    local_horizon_hours = 8760
    local_p_min_on = 450
    local_p_stb = 25
    local_c_start = 1500
    local_demand = 7

    local_operating_cases = ["ON_OFF", "ON_OFF_STANDBY"]
    local_case_segment_options = {
        "ON_OFF": [1, 12],
        "ON_OFF_STANDBY": [1, 4, 8, 12],
    }

    local_case_prefix = {
        "ON_OFF": "OO",
        "ON_OFF_STANDBY": "OOS",
    }

    local_time_data = pd.merge_asof(
        wind_df.sort_values("datetime"),
        price_df.sort_values("datetime"),
        on="datetime",
        direction="nearest",
        tolerance=pd.Timedelta("90min")
    )

    local_time_data = local_time_data.dropna(subset=["P_RE_kW", "price_buy"]).copy()
    local_time_data = local_time_data.sort_values("datetime").reset_index(drop=True)

    if local_time_data.empty:
        raise ValueError(
            "Merged local_time_data is empty. Check time zones or timestamps in Ninja and Energy-Charts files."
        )

    local_time_data = local_time_data.iloc[:local_horizon_hours].copy()

    # Required for this computational-comparison table:
    # export price equals import price.
    local_time_data["price_sell"] = local_time_data["price_buy"]
    local_time_data["demand_kg"] = local_demand

    local_T = len(local_time_data)
    local_p_re = local_time_data["P_RE_kW"].tolist()
    local_price_buy = local_time_data["price_buy"].tolist()
    local_price_sell = local_time_data["price_sell"].tolist()
    local_E_d_series = local_time_data["demand_kg"].tolist()
    local_M_grid = max(P_el_max, max(local_p_re))

    data_temp = data[data["Temperature_C"] == local_temperature].copy()
    if data_temp.empty:
        raise ValueError(f"Temperature {local_temperature} not found. Available temperatures: {temps}")

    data_temp["loss_curve"] = (1 - data_temp["Efficiency_eta"]) * data_temp["Power"]

    local_computational_results = []

    for operating_case in local_operating_cases:
        case_prefix = local_case_prefix[operating_case]

        for n_segments in local_case_segment_options[operating_case]:

            p_breaks = list(np.linspace(0.0, P_el_max, n_segments + 1))

            df_fit = (
                data_temp[["Power", "loss_curve"]]
                .sort_values("Power")
                .drop_duplicates("Power")
                .copy()
            )

            if df_fit["Power"].min() > 0:
                df_fit = pd.concat(
                    [
                        pd.DataFrame(
                            {
                                "Power": [0.0],
                                "loss_curve": [0.0]
                            }
                        ),
                        df_fit
                    ],
                    ignore_index=True
                ).sort_values("Power")

            loss_breaks = np.interp(
                p_breaks,
                df_fit["Power"],
                df_fit["loss_curve"]
            )

            s = []
            for i in range(len(p_breaks) - 1):
                dx = p_breaks[i + 1] - p_breaks[i]
                dy = loss_breaks[i + 1] - loss_breaks[i]
                if abs(dx) < 1e-9:
                    s.append(0.0)
                else:
                    s.append(dy / dx)

            negative_slopes = [val for val in s if val < -1e-9]
            if negative_slopes:
                print(
                    f"Warning: negative loss slopes at T={local_temperature}, "
                    f"operating_case={operating_case}, "
                    f"segments={n_segments}: {negative_slopes}"
                )

            delta_p = [
                p_breaks[i + 1] - p_breaks[i]
                for i in range(len(p_breaks) - 1)
            ]

            N_seg = len(s)

            m = gp.Model(
                f"computational_{operating_case}_T{local_temperature}_S{n_segments}_Pmin{local_p_min_on}"
            )

            m.setParam("OutputFlag", 0)
            m.setParam("TimeLimit", 3600)
            m.setParam("MIPGap", 0)
            m.setParam("MIPFocus", 2)
            m.setParam("Threads", 0)

            # Decision variables
            p_el = m.addVars(local_T, lb=0, ub=P_el_max, name="p_el")
            p_prod = m.addVars(local_T, lb=0, ub=P_el_max, name="p_prod")
            p_grid_buy = m.addVars(local_T, lb=0, name="p_grid_buy")
            p_sell = m.addVars(local_T, lb=0, name="p_sell")

            e = m.addVars(local_T, lb=e_min, ub=e_max, name="e")
            e_el = m.addVars(local_T, lb=0, name="e_el")
            loss = m.addVars(local_T, lb=0, name="loss")

            p_ti = m.addVars(local_T, N_seg, lb=0, name="p_ti")
            z = m.addVars(local_T, N_seg, vtype=GRB.BINARY, name="z")

            y_on = m.addVars(local_T, vtype=GRB.BINARY, name="y_on")
            y_stb = m.addVars(local_T, vtype=GRB.BINARY, name="y_stb")
            y_buy = m.addVars(local_T, vtype=GRB.BINARY, name="y_buy")
            y_start = m.addVars(local_T, vtype=GRB.BINARY, name="y_start")

            # Objective: unchanged structure.
            m.setObjective(
                gp.quicksum(C_hyo * e_el[t] for t in range(local_T))
                + gp.quicksum(
                    local_price_sell[t] * p_sell[t] * dt
                    for t in range(local_T)
                )
                - gp.quicksum(
                    local_price_buy[t] * p_grid_buy[t] * dt
                    for t in range(local_T)
                )
                - gp.quicksum(
                    local_c_start * y_start[t]
                    for t in range(local_T)
                ),
                GRB.MAXIMIZE
            )

            # Constraints: same model equations as the uploaded code.
            for t in range(local_T):

                m.addConstr(
                    loss[t] ==
                    gp.quicksum(
                        s[i] * p_ti[t, i]
                        for i in range(N_seg)
                    ),
                    name=f"loss_pwl_{t}"
                )

                for i in range(N_seg):
                    m.addConstr(
                        p_ti[t, i] <= delta_p[i] * z[t, i],
                        name=f"seg_bound_{t}_{i}"
                    )

                m.addConstr(
                    p_prod[t] ==
                    gp.quicksum(
                        p_ti[t, i]
                        for i in range(N_seg)
                    ),
                    name=f"prod_decomp_{t}"
                )

                for i in range(1, N_seg):
                    m.addConstr(
                        z[t, i] <= z[t, i - 1],
                        name=f"seq_seg_{t}_{i}"
                    )

                    m.addConstr(
                        p_ti[t, i - 1] >=
                        delta_p[i - 1] * z[t, i],
                        name=f"seg_contiguity_{t}_{i}"
                    )

                m.addConstr(
                    y_on[t] + y_stb[t] <= 1,
                    name=f"state_excl_{t}"
                )

                m.addConstr(
                    p_prod[t] <= P_el_max * y_on[t],
                    name=f"prod_max_{t}"
                )

                m.addConstr(
                    p_prod[t] >= local_p_min_on * y_on[t],
                    name=f"prod_min_{t}"
                )

                m.addConstr(
                    p_el[t] ==
                    p_prod[t] + local_p_stb * y_stb[t],
                    name=f"p_el_def_{t}"
                )

                m.addConstr(
                    e_el[t] ==
                    (p_prod[t] - loss[t]) * dt / HHV,
                    name=f"h2_prod_{t}"
                )

                m.addConstr(
                    loss[t] <= p_prod[t],
                    name=f"loss_bound_{t}"
                )

                m.addConstr(
                    p_grid_buy[t] + local_p_re[t] ==
                    p_el[t] + p_sell[t],
                    name=f"power_balance_{t}"
                )

                m.addConstr(
                    p_grid_buy[t] <= local_M_grid * y_buy[t],
                    name=f"buy_mode_{t}"
                )

                m.addConstr(
                    p_sell[t] <= local_M_grid * (1 - y_buy[t]),
                    name=f"sell_mode_{t}"
                )

                if t == 0:
                    m.addConstr(
                        e[0] ==
                        e_0 + e_el[0] - local_E_d_series[0],
                        name="storage_0"
                    )
                else:
                    m.addConstr(
                        e[t] ==
                        e[t - 1] + e_el[t] - local_E_d_series[t],
                        name=f"storage_{t}"
                    )

                add_operating_case_constraints(
                    m=m,
                    t=t,
                    case_name=operating_case,
                    y_on=y_on,
                    y_stb=y_stb
                )

                if t == 0:
                    m.addConstr(
                        y_start[0] == y_on[0],
                        name="startup_initial"
                    )
                else:
                    m.addConstr(
                        y_start[t] >=
                        y_on[t] -
                        y_on[t - 1] -
                        y_stb[t - 1],
                        name=f"startup_lb_{t}"
                    )

                    m.addConstr(
                        y_start[t] <= y_on[t],
                        name=f"startup_ub_on_{t}"
                    )

                    m.addConstr(
                        y_start[t] <= 1 - y_on[t - 1],
                        name=f"startup_ub_prev_on_{t}"
                    )

                    m.addConstr(
                        y_start[t] <= 1 - y_stb[t - 1],
                        name=f"startup_ub_prev_stb_{t}"
                    )

            # Terminal storage fairness constraint: unchanged.
            m.addConstr(e[local_T - 1] >= e_0, name="terminal_storage_min")

            m.optimize()

            runtime_seconds = float(m.Runtime)
            binary_count = count_binary_variables(m)
            has_solution = (m.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]) and (m.SolCount > 0)

            local_computational_results.append(
                {
                    "Case": f"{case_prefix}-{n_segments}",
                    "Operating_Case": operating_case,
                    "Segments": n_segments,
                    "Temperature_C": local_temperature,
                    "Horizon_hours": local_T,
                    "Computational_time_s": runtime_seconds,
                    "Binary_variables": binary_count,
                    "Binary_variables_formatted": format_binary_count(binary_count, local_T),
                    "Gurobi_Status": m.status,
                    "MIPGap": m.MIPGap if m.SolCount > 0 else None,
                    "Converged": bool(m.status == GRB.OPTIMAL or (m.SolCount > 0 and m.MIPGap <= 0.005)),
                    "Profit_EUR": m.objVal if has_solution else None,
                    "ObjBound_EUR": m.ObjBound if has_solution else None,
                }
            )

            print(
                f"Computational table case {case_prefix}-{n_segments}: "
                f"status={m.status}, runtime={runtime_seconds:.1f}s, "
                f"binary={format_binary_count(binary_count, local_T)}, "
                f"gap={m.MIPGap if m.SolCount > 0 else None}, "
                f"profit={m.objVal if has_solution else None}"
            )

    computational_OO_OOS_df = pd.DataFrame(local_computational_results)

    _build_computational_table_OO_OOS(
        computational_df=computational_OO_OOS_df,
        output_csv=computational_OO_OOS_out,
        output_latex=computational_OO_OOS_latex_out
    )

    return computational_OO_OOS_df


computational_OO_OOS_df = run_computational_aspects_OO_OOS()
